# EMA/SMA Crossover Strategy — VectorBT

In [1]:
import vectorbt as vbt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import schwabdev
import os
from dotenv import load_dotenv

## 1. Load Data

In [2]:
load_dotenv()

app_key = os.getenv("APP_KEY")
app_secret = os.getenv("APP_SECRET")

client = client = schwabdev.Client(app_key, app_secret)
data = client.price_history(symbol='NVDA', periodType='year', period=10, frequencyType='daily').json()['candles']
df = pd.DataFrame(data)

df['datetime'] = pd.to_datetime(df['datetime'], unit='ms').dt.strftime('%m-%d-%Y')
df.set_index(inplace=True, keys="datetime")

df.to_csv("crossover_data.csv", index=True)

The refresh token has expired!


[Schwabdev] Open to authenticate: https://api.schwabapi.com/v1/oauth/authorize?client_id=lCjVNk9AkOy4AhQFPcAIEUwIKl7Laih2iajNPvwyAc4861KZ&redirect_uri=https://127.0.0.1


No authorization URL provided, cannot continue.
The refresh token has expired!


[Schwabdev] Open to authenticate: https://api.schwabapi.com/v1/oauth/authorize?client_id=lCjVNk9AkOy4AhQFPcAIEUwIKl7Laih2iajNPvwyAc4861KZ&redirect_uri=https://127.0.0.1


No authorization URL provided, cannot continue.
The refresh token has expired!


[Schwabdev] Open to authenticate: https://api.schwabapi.com/v1/oauth/authorize?client_id=lCjVNk9AkOy4AhQFPcAIEUwIKl7Laih2iajNPvwyAc4861KZ&redirect_uri=https://127.0.0.1


No authorization URL provided, cannot continue.


JSONDecodeError: Expecting value: line 3 column 9 (char 22)

In [2]:
df = pd.read_csv(
    'crossover_data.csv',
    parse_dates=['datetime'],
    date_format='%m-%d-%Y',
    index_col='datetime'
)

close = df['close']   # vbt just needs the close price Series
print(close.tail())

datetime
2026-02-13    182.81
2026-02-17    184.97
2026-02-18    187.98
2026-02-19    187.90
2026-02-20    189.82
Name: close, dtype: float64


## 2. Compute Indicators & Signals

In [3]:
EMA_PERIOD = 20
SMA_PERIOD = 50

# vbt.MA.run returns an MA accessor; .ma gives back the Series
ema = vbt.MA.run(close, window=EMA_PERIOD, ewm=True).ma     # ewm=True → EMA
sma = vbt.MA.run(close, window=SMA_PERIOD, ewm=False).ma    # ewm=False → SMA

# Crossover signals:
#   Entry  = EMA crosses ABOVE SMA  (previous bar EMA <= SMA, current bar EMA > SMA)
#   Exit   = EMA crosses BELOW SMA
entries = (ema > sma) & (ema.shift(1) <= sma.shift(1))
exits   = (ema < sma) & (ema.shift(1) >= sma.shift(1))

print(f"Total entry signals : {entries.sum()}")
print(f"Total exit  signals : {exits.sum()}")
print(ema)

Total entry signals : 3
Total exit  signals : 2
datetime
2025-02-20           NaN
2025-02-21           NaN
2025-02-24           NaN
2025-02-25           NaN
2025-02-26           NaN
                 ...    
2026-02-13    185.319132
2026-02-17    185.285881
2026-02-18    185.542464
2026-02-19    185.766991
2026-02-20    186.152992
Name: (20, True, close), Length: 252, dtype: float64


## 3. Build the Portfolio
`vbt.Portfolio.from_signals()` is the main entry point for signal-based strategies.

### Fees
VectorBT supports fees via the `fees` parameter — it's a **fraction of trade value**
(not per share). Your backtrader code used `commission=0.01` which in backtrader's
default mode is also a fraction (1%), so we pass `fees=0.01` directly.

### Slippage
VectorBT supports slippage via `slippage` — also a fraction of price, applied to
fill price. Your backtrader code used `set_slippage_perc(0.005)` = 0.5%, so we
pass `slippage=0.005`. VectorBT applies it as: fill_price = signal_price * (1 + slippage)
for buys and * (1 - slippage) for sells — same behavior as backtrader's percent slippage.

### Size
backtrader used `SizerFix(stake=10)` meaning 10 shares per trade. VectorBT uses
`size=10` for a fixed share count.

In [ ]:
INITIAL_CASH  = 10_000.0
STAKE         = 10          # shares per trade
FEES          = 0.01        # 1% of trade value (matches backtrader commission=0.01)
SLIPPAGE      = 0.005       # 0.5% fill-price slippage (matches set_slippage_perc(0.005))

pf = vbt.Portfolio.from_signals(
    close,
    entries=entries,
    exits=exits,
    size=STAKE,
    size_type='shares',       # fixed share count, not % of capital
    init_cash=INITIAL_CASH,
    fees=FEES,
    slippage=SLIPPAGE,
    freq='D'                  # daily data — needed for annualisation in stats
)

print(f"Starting Value : ${INITIAL_CASH:,.2f}")
print(f"Ending Value   : ${pf.final_value():.2f}")

## 4. Performance Stats
`pf.stats()` replaces all your separate analyzers (`TimeReturn`, `TradeAnalyzer`, etc.).
It returns a single Series with ~30 metrics including Sharpe, Sortino, Max Drawdown,
Win Rate, Profit Factor, and more — all in one call.

In [ ]:
stats = pf.stats()
print(stats)

## 5. Trade-Level Analysis
`pf.trades` is a `Trades` accessor — equivalent to backtrader's `TradeAnalyzer`.
You get a full DataFrame of every trade with entry/exit price, PnL, duration, etc.

In [ ]:
# Full trade log
trade_records = pf.trades.records_readable
print(trade_records)

# Quick summary
print("\n--- Trade Summary ---")
print(f"Total trades  : {pf.trades.count()}")
print(f"Win rate      : {pf.trades.win_rate():.1%}")
print(f"Avg PnL/trade : ${pf.trades.pnl.mean():.2f}")
print(f"Profit factor : {pf.trades.profit_factor():.2f}")

## 6. Returns & Equity Curve
Equivalent to backtrader's `TimeReturn` analyzer and your manual `equity_curve` list.

In [ ]:
# Daily returns Series
returns = pf.returns()

# Equity curve (portfolio value over time)
equity = pf.value()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

equity.plot(ax=axes[0], title='Equity Curve', color='steelblue')
axes[0].set_ylabel('Portfolio Value ($)')

returns.plot(ax=axes[1], title='Daily Returns', color='gray', alpha=0.7)
axes[1].set_ylabel('Return')
axes[1].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('crossover_equity_vectorbt.png', dpi=150)
plt.show()

## 7. Drawdown
Equivalent to backtrader's `DrawDown` observer.

In [ ]:
pf.drawdown().plot(title='Drawdown (%)')
plt.savefig('crossover_drawdown_vectorbt.png', dpi=150)
plt.show()

print(f"Max Drawdown: {pf.max_drawdown():.2%}")

## 8. Full Interactive Plot (replaces `cerebro.plot()`)
VectorBT uses plotly under the hood, so this is interactive in Jupyter.

In [ ]:
pf.plot().show()

## 9. BONUS — Parameter Optimization (free with VectorBT)
One of VectorBT's killer features: you can sweep parameter grids and run ALL combinations
simultaneously in one vectorized call. In backtrader you'd need `cerebro.optstrategy()`
which spawns separate processes. Here it's just array broadcasting.

In [ ]:
ema_windows = [10, 20, 30]
sma_windows = [40, 50, 60, 100]

# vbt.MA.run accepts lists → returns a multi-column DataFrame (one col per param combo)
ema_all = vbt.MA.run(close, window=ema_windows, ewm=True,  param_product=True)
sma_all = vbt.MA.run(close, window=sma_windows, ewm=False, param_product=True)

# Broadcast: entries/exits become DataFrames with one column per (ema, sma) combo
entries_all = (ema_all.ma > sma_all.ma) & (ema_all.ma.shift(1) <= sma_all.ma.shift(1))
exits_all   = (ema_all.ma < sma_all.ma) & (ema_all.ma.shift(1) >= sma_all.ma.shift(1))

pf_all = vbt.Portfolio.from_signals(
    close,
    entries=entries_all,
    exits=exits_all,
    size=STAKE,
    size_type='shares',
    init_cash=INITIAL_CASH,
    fees=FEES,
    slippage=SLIPPAGE,
    freq='D'
)

# Stats for every combination — returns a DataFrame
opt_stats = pf_all.stats()

# Find best Sharpe ratio
best_idx = opt_stats['Sharpe Ratio'].idxmax()
print(f"Best params by Sharpe: {best_idx}")
print(opt_stats.loc[best_idx])